# ECR Experiment Notebook

Notebook simple pour reproduire une experience alignee avec l'article **Towards Empathetic Conversational Recommender Systems**.

- EDA complete
- Evaluation des resultats
- Comparaison des modeles avec visualisations externes

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.viz.plots import (
    plot_feedback_distribution,
    plot_model_rankings,
    plot_objective_metrics,
    plot_subjective_metrics,
    plot_subjective_radar,
)

In [2]:
def get_config():
    return {
        "results_dir": ROOT / "results",
        "objective_file": "objective_metrics.csv",
        "subjective_llm_file": "subjective_metrics_llm.csv",
        "subjective_human_file": "subjective_metrics_human.csv",
    }

In [3]:
def load_results_data(config):
    results_dir = config["results_dir"]
    objective_path = results_dir / config["objective_file"]
    subjective_llm_path = results_dir / config["subjective_llm_file"]
    subjective_human_path = results_dir / config["subjective_human_file"]

    if objective_path.exists() and subjective_llm_path.exists() and subjective_human_path.exists():
        df_obj = pd.read_csv(objective_path)
        df_subj_llm = pd.read_csv(subjective_llm_path)
        df_subj_human = pd.read_csv(subjective_human_path)
    else:
        # Fallback: valeurs du papier pour execution immediate.
        df_obj = pd.DataFrame([
            {"Model": "KBRD", "AUC": 0.503, "RT@1": 0.040, "RT@10": 0.182, "RT@50": 0.381, "R@1": 0.037, "R@10": 0.175, "R@50": 0.360},
            {"Model": "KGSF", "AUC": 0.513, "RT@1": 0.043, "RT@10": 0.195, "RT@50": 0.383, "R@1": 0.040, "R@10": 0.182, "R@50": 0.361},
            {"Model": "RevCore", "AUC": 0.514, "RT@1": 0.054, "RT@10": 0.230, "RT@50": 0.410, "R@1": 0.046, "R@10": 0.209, "R@50": 0.390},
            {"Model": "UCCR", "AUC": 0.499, "RT@1": 0.038, "RT@10": 0.208, "RT@50": 0.423, "R@1": 0.039, "R@10": 0.198, "R@50": 0.407},
            {"Model": "UniCRS", "AUC": 0.506, "RT@1": 0.052, "RT@10": 0.229, "RT@50": 0.439, "R@1": 0.047, "R@10": 0.212, "R@50": 0.414},
            {"Model": "ECR", "AUC": 0.541, "RT@1": 0.055, "RT@10": 0.238, "RT@50": 0.452, "R@1": 0.049, "R@10": 0.220, "R@50": 0.428},
        ])

        df_subj_llm = pd.DataFrame([
            {"Model": "UniCRS", "Emo Int": 0.400, "Emo Pers": 0.942, "Log Pers": 0.793, "Info": 0.673, "Life": 2.241},
            {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 1.706, "Emo Pers": 3.043, "Log Pers": 3.474, "Info": 2.975, "Life": 4.182},
            {"Model": "GPT-3.5-turbo", "Emo Int": 2.215, "Emo Pers": 3.754, "Log Pers": 4.782, "Info": 4.147, "Life": 5.338},
            {"Model": "Llama 2-7B-Chat", "Emo Int": 3.934, "Emo Pers": 6.030, "Log Pers": 5.886, "Info": 5.904, "Life": 7.129},
            {"Model": "ECR[DialoGPT]", "Emo Int": 4.011, "Emo Pers": 4.878, "Log Pers": 4.736, "Info": 5.094, "Life": 5.906},
            {"Model": "ECR[Llama 2-Chat]", "Emo Int": 6.826, "Emo Pers": 7.724, "Log Pers": 6.702, "Info": 7.653, "Life": 8.063},
        ])

        df_subj_human = pd.DataFrame([
            {"Model": "UniCRS", "Emo Int": 0.947, "Emo Pers": 0.775, "Log Pers": 1.158, "Info": 0.380, "Life": 1.805},
            {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 2.048, "Emo Pers": 2.555, "Log Pers": 3.265, "Info": 1.822, "Life": 3.648},
            {"Model": "GPT-3.5-turbo", "Emo Int": 2.890, "Emo Pers": 3.678, "Log Pers": 5.323, "Info": 3.233, "Life": 5.125},
            {"Model": "Llama 2-7B-Chat", "Emo Int": 4.432, "Emo Pers": 6.152, "Log Pers": 6.393, "Info": 5.713, "Life": 7.463},
            {"Model": "ECR[DialoGPT]", "Emo Int": 5.097, "Emo Pers": 4.817, "Log Pers": 5.398, "Info": 4.628, "Life": 6.385},
            {"Model": "ECR[Llama 2-Chat]", "Emo Int": 7.130, "Emo Pers": 7.575, "Log Pers": 7.403, "Info": 7.172, "Life": 8.468},
        ])

    return df_obj, df_subj_llm, df_subj_human

In [4]:
# Import des donnees en une seule cellule
cfg = get_config()
df_obj, df_subj_llm, df_subj_human = load_results_data(cfg)

df_obj.head()

,Model,AUC,RT@1,RT@10,RT@50,R@1,R@10,R@50
0,KBRD,0.503,0.040,0.182,0.381,0.037,0.175,0.360
1,KGSF,0.513,0.043,0.195,0.383,0.040,0.182,0.361
2,RevCore,0.514,0.054,0.230,0.410,0.046,0.209,0.390
3,UCCR,0.499,0.038,0.208,0.423,0.039,0.198,0.407
4,UniCRS,0.506,0.052,0.229,0.439,0.047,0.212,0.414


In [5]:
def explore_objective_data(df_obj):
    print("=== OBJECTIVE DATA ===")
    print("Shape:", df_obj.shape)
    print("Models:", df_obj["Model"].tolist())
    print("\nDescriptive statistics:")
    print(df_obj.describe(include="all"))

    # Distribution de feedback (article: like 81.1, dislike 4.9, not say 14.0)
    df_feedback = pd.DataFrame({
        "feedback": ["like", "dislike", "not say"],
        "count": [8110, 490, 1400],
    })
    return df_feedback

In [6]:
def explore_subjective_data(df_subj_llm, df_subj_human):
    print("=== SUBJECTIVE DATA (LLM SCORER) ===")
    print("Shape:", df_subj_llm.shape)
    print(df_subj_llm.describe(include="all"))
    print("\n=== SUBJECTIVE DATA (HUMAN) ===")
    print("Shape:", df_subj_human.shape)
    print(df_subj_human.describe(include="all"))

In [7]:
def evaluate_results(df_obj, df_subj_llm, df_subj_human):
    best_auc = df_obj.loc[df_obj["AUC"].idxmax(), ["Model", "AUC"]]
    best_rt10 = df_obj.loc[df_obj["RT@10"].idxmax(), ["Model", "RT@10"]]
    best_life_llm = df_subj_llm.loc[df_subj_llm["Life"].idxmax(), ["Model", "Life"]]
    best_life_human = df_subj_human.loc[df_subj_human["Life"].idxmax(), ["Model", "Life"]]

    summary = pd.DataFrame([
        {"metric": "best_auc", "model": best_auc["Model"], "value": best_auc["AUC"]},
        {"metric": "best_rt10", "model": best_rt10["Model"], "value": best_rt10["RT@10"]},
        {"metric": "best_life_llm", "model": best_life_llm["Model"], "value": best_life_llm["Life"]},
        {"metric": "best_life_human", "model": best_life_human["Model"], "value": best_life_human["Life"]},
    ])
    return summary

In [8]:
def build_comparison_table(df_obj, df_subj_llm, df_subj_human):
    llm = df_subj_llm.add_suffix("_llm").rename(columns={"Model_llm": "Model"})
    human = df_subj_human.add_suffix("_human").rename(columns={"Model_human": "Model"})
    comparison = df_obj.merge(llm, on="Model", how="left").merge(human, on="Model", how="left")
    return comparison

In [9]:
def run_eda_and_summary(df_obj, df_subj_llm, df_subj_human):
    df_feedback = explore_objective_data(df_obj)
    explore_subjective_data(df_subj_llm, df_subj_human)
    summary = evaluate_results(df_obj, df_subj_llm, df_subj_human)
    comparison = build_comparison_table(df_obj, df_subj_llm, df_subj_human)
    return df_feedback, summary, comparison

In [10]:
# Derniere cellule: compilation des donnees + visualisations comparatives
df_feedback, summary_df, comparison_df = run_eda_and_summary(df_obj, df_subj_llm, df_subj_human)

display(summary_df)
display(comparison_df.head())

plot_feedback_distribution(df_feedback)
plot_model_rankings(df_obj, metric="AUC")
plot_objective_metrics(df_obj)
plot_subjective_metrics(df_subj_llm, "Subjective Metrics - LLM Scorer")
plot_subjective_metrics(df_subj_human, "Subjective Metrics - Human Annotators")
plot_subjective_radar(df_subj_llm, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (LLM Scorer)")
plot_subjective_radar(df_subj_human, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (Human)")

=== OBJECTIVE DATA ===
Shape: (6, 8)
Models: ['KBRD', 'KGSF', 'RevCore', 'UCCR', 'UniCRS', 'ECR']

Descriptive statistics:
       Model       AUC      RT@1     RT@10     RT@50      R@1      R@10  \
count      6  6.000000  6.000000  6.000000  6.000000  6.00000  6.000000   
unique     6       NaN       NaN       NaN       NaN      NaN       NaN   
top     KBRD       NaN       NaN       NaN       NaN      NaN       NaN   
freq       1       NaN       NaN       NaN       NaN      NaN       NaN   
mean     NaN  0.512667  0.047000  0.213667  0.414667  0.04300  0.199333   
std      NaN  0.015029  0.007537  0.022259  0.029029  0.00494  0.017750   
min      NaN  0.499000  0.038000  0.182000  0.381000  0.03700  0.175000   
25%      NaN  0.503750  0.040750  0.198250  0.389750  0.03925  0.186000   
50%      NaN  0.509500  0.047500  0.218500  0.416500  0.04300  0.203500   
75%      NaN  0.513750  0.053500  0.229750  0.435000  0.04675  0.211250   
max      NaN  0.541000  0.055000  0.238000  0.452000

,metric,model,value
0,best_auc,ECR,0.541
1,best_rt10,ECR,0.238
2,best_life_llm,ECR[Llama 2-Chat],8.063
3,best_life_human,ECR[Llama 2-Chat],8.468


,Model,AUC,RT@1,RT@10,RT@50,R@1,R@10,R@50,Emo Int_llm,Emo Pers_llm,Log Pers_llm,Info_llm,Life_llm,Emo Int_human,Emo Pers_human,Log Pers_human,Info_human,Life_human
0,KBRD,0.503,0.040,0.182,0.381,0.037,0.175,0.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KGSF,0.513,0.043,0.195,0.383,0.040,0.182,0.361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RevCore,0.514,0.054,0.230,0.410,0.046,0.209,0.390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,UCCR,0.499,0.038,0.208,0.423,0.039,0.198,0.407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,UniCRS,0.506,0.052,0.229,0.439,0.047,0.212,0.414,0.4,0.942,0.793,0.673,2.241,0.947,0.775,1.158,0.38,1.805


/workspace/src/viz/plots.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_feedback, x="feedback", y="count", ax=ax, palette="muted")
/workspace/src/viz/plots.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=ranked, x="Model", y=metric, ax=ax, palette="crest")
